# **Part #1:**

In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# Import original Olist data file from HW2 with features for model training

olist_orders_features = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DATA 6545 Data Science and MLOps/Data Files/HW3/Olist_Orders_Model_Dataset.csv')
olist_orders_features = olist_orders_features.drop(columns=['Unnamed: 0'])
olist_orders_features.head()

,delivery_days,delivery_vs_estimated,price,freight_ratio,product_category,customer_state,seller_state,payment_type,distance_miles,product_volume_cm^3,total_order_value,is_positive_review
0,8.0,-8.0,29.99,0.290764,housewares,SP,SP,credit_card,11.513378,1976.0,89.97,1
1,8.0,-8.0,29.99,0.290764,housewares,SP,SP,voucher,11.513378,1976.0,89.97,1
2,8.0,-8.0,29.99,0.290764,housewares,SP,SP,voucher,11.513378,1976.0,89.97,1
3,13.0,-6.0,118.70,0.191744,perfumery,BA,SP,boleto,525.344592,4693.0,118.70,1
4,9.0,-18.0,159.90,0.120200,auto,GO,SP,credit_card,318.202316,9576.0,159.90,1


In [3]:
# Original model from HW2:

# Train/Test Split:
from sklearn.model_selection import train_test_split

X = olist_orders_features.drop(columns=['is_positive_review'])
y = olist_orders_features['is_positive_review']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'Train balance: {y_train.mean():.1%} positive')
print(f'Test balance: {y_test.mean():.1%} positive')

X_train shape: (90807, 11)
X_test shape: (22702, 11)
Train balance: 76.2% positive
Test balance: 76.2% positive


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

In [5]:
from sklearn.ensemble import RandomForestClassifier

random_forest_balanced = Pipeline([('preprocessor', preprocessor), ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))])

random_forest_balanced.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['delivery_days',
                                                   'delivery_vs_estimated',
                                                   'price', 'freight_ratio',
                                                   'distance_miles',
                                                   'product_volume_cm^3',
                                                   'total_order_value']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['product_category',
                                                   'customer_state',
                                                   'seller_state',
                                                   'payment_type'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [6]:
# Hyperparameter Tuning

from sklearn.model_selection import RandomizedSearchCV

parameter_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 20, 100],
    'classifier__min_samples_split': [2, 5, 10]
}

random_search = RandomizedSearchCV(random_forest_balanced, param_distributions=parameter_grid, n_iter=10, scoring='recall', cv=5, n_jobs=-1, random_state=42)

random_fit = random_search.fit(X_train, y_train)

In [7]:
# Tuned Model Evaluation

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

random_forest_tuned = random_fit.best_estimator_

y_pred_tuned = random_forest_tuned.predict(X_test)
y_prob_tuned = random_forest_tuned.predict_proba(X_test)[:, 1]

accuracy_HW2 = accuracy_score(y_test, y_pred_tuned)
precision_HW2 = precision_score(y_test, y_pred_tuned)
recall_HW2 = recall_score(y_test, y_pred_tuned)
f1_score_HW2 = f1_score(y_test, y_pred_tuned)
roc_auc_score_HW2 = roc_auc_score(y_test, y_prob_tuned)

print(f'Accuracy: {accuracy_HW2:.1%}')
print(f'Precision: {precision_HW2:.1%}')
print(f'Recall: {recall_HW2:.1%}')
print(f'f1 score: {f1_score_HW2:.1%}')
print(f'ROC-AUC score: {roc_auc_score_HW2:.1%}')

Accuracy: 85.0%
Precision: 85.3%
Recall: 97.0%
f1 score: 90.8%
ROC-AUC score: 79.0%


In [25]:
# Import full Olist data file from HW2 with all columns

olist_orders_full = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/DATA 6545 Data Science and MLOps/Data Files/HW4/Olist_Orders_Full.csv')
olist_orders_full = olist_orders_full.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'])
olist_orders_full.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customers_geolocation_lat_mean,customers_geolocation_lng_mean,delivery_days,delivery_vs_estimated,seller_state_match,freight_ratio,is_positive_review,distance_miles,product_volume_cm^3,total_order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,-23.577482,-46.587077,8.0,-8.0,True,0.290764,1,11.513378,1976.0,89.97
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,-23.577482,-46.587077,8.0,-8.0,True,0.290764,1,11.513378,1976.0,89.97
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,-23.577482,-46.587077,8.0,-8.0,True,0.290764,1,11.513378,1976.0,89.97
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,-12.186877,-44.540232,13.0,-6.0,False,0.191744,1,525.344592,4693.0,118.70
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,-16.745150,-48.514783,9.0,-18.0,False,0.120200,1,318.202316,9576.0,159.90


In [26]:
# Handling null values in 'review_comment_message' column

olist_orders_drop = olist_orders_full.dropna(subset=['review_comment_message'])

olist_orders_drop.shape

(50245, 60)

In [8]:
# Install HuggingFace Transformers Library
!pip install transformers


In [11]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [27]:
olist_orders_sample = olist_orders_full.sample(n=500, random_state=42)

olist_orders_sample.shape

(500, 60)

In [28]:
import numpy as np

def sentiment_prediction(text):
    try:
        result = sentiment(text[:1000])[0]
        stars = int(result["label"].split()[0])
        return stars
    except:
        return np.nan

olist_orders_sample['predicted_stars'] = olist_orders_sample['review_comment_message'].apply(sentiment_prediction)

olist_orders_sample = olist_orders_sample.dropna(subset=['predicted_stars'])

olist_orders_sample['foundation_pred'] = olist_orders_sample['predicted_stars'].apply(lambda x: 1 if x >= 4 else 0)

olist_orders_sample['foundation_pred']

,foundation_pred
48018,0
62675,0
53195,1
28650,1
9700,1
...,...
77633,0
114504,1
66066,0
20899,1


In [29]:
# Foundation model performance

from sklearn.metrics import confusion_matrix

total = len(olist_orders_sample)
positive = olist_orders_sample['is_positive_review'].sum()
negative = total - positive

print(f'% Positive: {(positive / total) * 100}')
print(f'% Negative: {(negative / total) * 100}')

y_test_f = olist_orders_sample['is_positive_review']
y_pred_f = olist_orders_sample['foundation_pred']

cm_f = confusion_matrix(y_test_f, y_pred_f)
print(f'Confusion Matrix: \n{cm_f}')

% Positive: 67.72727272727272
% Negative: 32.27272727272727
Confusion Matrix: 
[[ 67   4]
 [ 36 113]]


In [38]:
olist_orders_sample = olist_orders_sample.rename(columns={'product_category_name': 'product_category'})

In [42]:
# Model performance comparison

column_names = X_test.columns.tolist()
column_names

rf_model_preds = random_forest_tuned.predict(olist_orders_sample[column_names])

olist_orders_sample['rf_model_pred'] = rf_model_preds

def metrics(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    }

foundation_metrics = metrics(olist_orders_sample['is_positive_review'], olist_orders_sample['foundation_pred'])
rf_metrics = metrics(olist_orders_sample['is_positive_review'], olist_orders_sample['rf_model_pred'])

comparison_df = pd.DataFrame([rf_metrics, foundation_metrics], index=['Random Forest Model', 'Foundation Model'])

print(comparison_df)

                     Accuracy  Precision    Recall  F1 Score
Random Forest Model  0.845455   0.821229  0.986577  0.896341
Foundation Model     0.818182   0.965812  0.758389  0.849624


Overall, the random forest model I created in HW2 performs better compared to the foundation model on the same data set. The random forest model has a higher accuracy and F1 score while outperforming the foundation model in recall, although the foundation model outperforms the random forest model in precision, but the trade-off between the two is greater with the foundation model. One of the advantages of having a foundation model is that it is easier to implement given that it is already programmed to interpret and predict data in a specific way. This makes it suitable for situations where you need a solution implemented quickly and don't have time to fully validate the results (given that the specific foundation model has good reputability). A foundation model may not allow for the same flexibility for modifying the prediction outcomes to your specific use-case, however, and given that this model has a low recall but high precision (which is the opposite of what is optimal for this specific business problem), it would be useful here to have greater flexibility to make adjustments to these metrics.